# Juego de Monedas Ocultas — Aprendizaje por Refuerzo

**Problema:** Un agente debe encontrar monedas escondidas en un tablero. Cada casilla tiene una probabilidad fija (desconocida) de contener una moneda. El agente debe aprender qué casillas son mejores mediante prueba y error.

**Elementos de Aprendizaje por Refuerzo:**
- **Estado**: la casilla que el agente elige cavar (0 a N-1)
- **Entorno**: tablero con monedas ocultas, cada casilla con probabilidad $p_i$ de dar recompensa
- **Acción**: cavar en una casilla específica
- **Política**: estrategia para elegir qué casilla cavar (ε-greedy, UCB o Gradiente)
- **Recompensa**: +1 si hay moneda, 0 si no
- **Función de valor**: $Q(casilla)$ = valor estimado de la casilla

In [ ]:
import numpy as np
import math
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm

np.random.seed(42)

ModuleNotFoundError: No module named 'tqdm'

---
## 1. Entorno: Tablero

El tablero contiene monedas ocultas. Cada casilla tiene una **probabilidad real** $p_i$ de tener una moneda. Cuando el agente cava en una casilla, recibe +1 si hay moneda, 0 si no.

In [ ]:
class Tablero():
    def __init__(self, num_casillas=25, prob_min=0, prob_max=0.5):
        # Numero total de casillas en el tablero
        self.num_casillas = num_casillas
        # Cada casilla tiene una probabilidad REAL (fija) de tener moneda
        # Estas probabilidades son desconocidas para el agente
        self.probabilidades = np.random.uniform(prob_min, prob_max, num_casillas)
        # La mejor casilla es la que tiene mayor probabilidad
        self.mejor_casilla = np.argmax(self.probabilidades)

    def cavar(self, casilla):
        # Devuelve 1 si hay moneda en la casilla, 0 si no
        if np.random.uniform() < self.probabilidades[casilla]:
            return 1
        return 0

    def mostrar(self):
        print("Probabilidades reales de cada casilla:")
        for i in range(self.num_casillas):
            print(f"  Casilla {i}: {self.probabilidades[i]:.3f}", end="")
            if i == self.mejor_casilla:
                print(" ← MEJOR", end="")
            print()

---
## 2. Agente

El agente aprende una **función de valor** $Q(casilla)$ que estima qué tan buena es cada casilla. Usa esta función para decidir dónde cavar.

**Estrategia de exploración/explotación:**
- **Exploración (ε-greedy)**: con probabilidad ε, elige una casilla al azar
- **Explotación**: elige la casilla con mayor valor Q conocido

**Actualización:** al final de cada episodio, recorre las acciones en orden inverso actualizando Q con:
$$Q(S_t) \leftarrow Q(S_t) + \alpha [R - Q(S_t)]$$

In [ ]:
class Agente():
    def __init__(self, tasa_aprendizaje=0.5, prob_exploracion=0.5):
        # Funcion de valor: diccionario que mapea casilla -> valor estimado
        self.funcion_de_valor = {}
        # Tasa de aprendizaje (alpha): que tan rapido se actualizan los valores
        self.tasa_aprendizaje = tasa_aprendizaje
        # Probabilidad de exploracion (epsilon): elegir casilla al azar
        self.prob_exploracion = prob_exploracion
        # Historial de casillas elegidas en el episodio actual
        self.casillas_elegidas = []

    def reiniciar(self):
        # Limpia el historial para un nuevo episodio
        self.casillas_elegidas = []

    def elegir_casilla(self, tablero, explorar=True):
        # Elige que casilla cavar
        num_casillas = tablero.num_casillas

        # --- EXPLORACION: elegir al azar ---
        if explorar and np.random.uniform(0, 1) < self.prob_exploracion:
            return np.random.randint(num_casillas)

        # --- EXPLOTACION: elegir la casilla con mayor valor Q ---
        mejor_valor = -1e9
        mejor_casilla = 0
        for j in range(num_casillas):
            valor = 0 if self.funcion_de_valor.get(j) is None else self.funcion_de_valor.get(j)
            if valor >= mejor_valor:
                mejor_valor = valor
                mejor_casilla = j
        return mejor_casilla

    def actualizar(self, casilla):
        # Guarda la casilla elegida en el historial del episodio
        self.casillas_elegidas.append(casilla)

    def recompensa(self, valor_recompensa):
        # Actualiza la funcion de valor en orden inverso
        # Los valores se propagan hacia atras:
        #   - La ultima casilla recibe la recompensa directa
        #   - Las anteriores aprenden del valor de la siguiente
        for casilla in reversed(self.casillas_elegidas):
            if self.funcion_de_valor.get(casilla) is None:
                self.funcion_de_valor[casilla] = 0
            # Q(s) = Q(s) + alpha * (recompensa - Q(s))
            self.funcion_de_valor[casilla] += self.tasa_aprendizaje * (valor_recompensa - self.funcion_de_valor[casilla])
            # El valor actualizado pasa a ser la "recompensa" para la casilla anterior
            valor_recompensa = self.funcion_de_valor[casilla]

---
## 3. Juego

El juego conecta al agente con el tablero. Cada episodio:
1. El agente elige una casilla
2. Cava y obtiene recompensa
3. Repite hasta agotar los turnos
4. Al final, actualiza la función de valor con la recompensa acumulada

In [ ]:
class Juego():
    def __init__(self, agente, tablero, turnos_por_episodio=10):
        self.agente = agente
        self.tablero = tablero
        self.turnos_por_episodio = turnos_por_episodio

    def jugar_episodio(self):
        # Reinicia el historial del agente
        self.agente.reiniciar()

        recompensa_total = 0

        for _ in range(self.turnos_por_episodio):
            # El agente elige una casilla (con exploracion)
            casilla = self.agente.elegir_casilla(self.tablero, explorar=True)

            # Cava en la casilla y obtiene recompensa
            recompensa = self.tablero.cavar(casilla)
            recompensa_total += recompensa

            # Guarda la casilla en el historial
            self.agente.actualizar(casilla)

        # Al final del episodio, actualiza la funcion de valor con la recompensa total
        # Normalizamos dividiendo entre el maximo posible para que quede entre 0 y 1
        recompensa_normalizada = recompensa_total / self.turnos_por_episodio
        self.recompensa(recompensa_normalizada)

        return recompensa_total

    def autoentrenar(self, episodios=10000):
        recompensas = []
        for i in tqdm(range(1, episodios + 1)):
            recompensa = self.jugar_episodio()
            recompensas.append(recompensa)
        return recompensas

    def recompensa(self, valor):
        self.agente.recompensa(valor)

---
## 4. Entrenamiento del agente (con clases)

Creamos el tablero, el agente y lo entrenamos durante miles de episodios. Cada episodio el agente cava varias veces y aprende de la experiencia.

In [ ]:
# Creamos el tablero de 25 casillas (5x5)
tablero = Tablero(num_casillas=25)
print("=== TABLERO DE MONEDAS ===")
tablero.mostrar()

# Creamos el agente
agente = Agente(tasa_aprendizaje=0.3, prob_exploracion=0.2)

# Creamos el juego
juego = Juego(agente, tablero, turnos_por_episodio=10)

# Entrenamos
print("\nEntrenando al agente...")
recompensas = juego.autoentrenar(10000)
print("\nEntrenamiento completado.")

In [ ]:
# Mostramos resultados del entrenamiento
print(f"Recompensa promedio por episodio: {np.mean(recompensas[-1000:]):.3f}")
print(f"Recompensa maxima posible por episodio: {juego.turnos_por_episodio}")
print(f"\nValores Q aprendidos por el agente:")
for casilla in range(tablero.num_casillas):
    valor = agente.funcion_de_valor.get(casilla, 0)
    prob_real = tablero.probabilidades[casilla]
    print(f"  Casilla {casilla:2d}: Q={valor:.3f}  (prob real={prob_real:.3f})")

print(f"\nMejor casilla segun el agente: {max(agente.funcion_de_valor, key=agente.funcion_de_valor.get)}")
print(f"Mejor casilla real: {tablero.mejor_casilla}")

### Gráfica del aprendizaje

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(recompensas)
plt.xlabel('Episodio')
plt.ylabel('Recompensa total')
plt.title('Recompensa por episodio (cruda)')
plt.grid(True)

plt.subplot(1, 2, 2)
ventana = 200
suave = np.convolve(recompensas, np.ones(ventana)/ventana, mode='valid')
plt.plot(suave)
plt.xlabel('Episodio')
plt.ylabel('Recompensa promedio')
plt.title(f'Recompensa (media móvil {ventana})')
plt.grid(True)

plt.tight_layout()
plt.show()

---
## 5. Experimentos: Exploración vs Explotación

Ahora aplicamos los métodos de `02_bandits.ipynb` para comparar diferentes estrategias de exploración, usando la misma estructura de **bucles `for`**.

Cada "turno" es una cavada en una casilla. La recompensa es inmediata (1 si hay moneda, 0 si no).

### Experimento 1: ε-greedy

Comparamos diferentes valores de ε (probabilidad de explorar al azar).

```
for partida in range(partidas):
    for i, eps in enumerate(epsilons):
        for turno in range(turnos):
            # elegir casilla (explorar o explotar)
            # cavar y obtener recompensa
            # actualizar Q
```

In [ ]:
partidas = 500
turnos = 200
epsilons = [0, 0.05, 0.1, 0.2]

recompensas_eps = np.zeros((len(epsilons), turnos))
optimas_eps = np.zeros((len(epsilons), turnos))

for partida in range(partidas):
    for i, eps in enumerate(epsilons):
        Q = {k: 0 for k in range(tablero.num_casillas)}
        N = {k: 0 for k in range(tablero.num_casillas)}

        for turno in range(turnos):
            # Seleccionar accion
            if np.random.uniform() < eps:
                casilla = np.random.randint(tablero.num_casillas)
            else:
                max_q = -1e9
                for j in range(tablero.num_casillas):
                    if Q[j] > max_q:
                        max_q = Q[j]
                        casilla = j

            # Cavar y obtener recompensa
            N[casilla] += 1
            recompensa = tablero.cavar(casilla)

            # Actualizar Q (promedio incremental)
            Q[casilla] += (recompensa - Q[casilla]) / N[casilla]

            recompensas_eps[i][turno] += recompensa
            optimas_eps[i][turno] += (1 if casilla == tablero.mejor_casilla else 0)

recompensas_eps /= partidas
optimas_eps /= partidas

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for i, eps in enumerate(epsilons):
    plt.plot(recompensas_eps[i], label=f'ε = {eps}')
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Recompensa promedio')
plt.title('ε-greedy: Recompensa')

plt.subplot(1, 2, 2)
for i, eps in enumerate(epsilons):
    plt.plot(optimas_eps[i], label=f'ε = {eps}')
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Proporción óptima')
plt.title('ε-greedy: Acción óptima')
plt.tight_layout()
plt.show()

### Experimento 2: UCB (Upper Confidence Bound)

$$A_t = \underset{a}{\arg\max} \left[ Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} \right]$$

UCB explora casillas con **alta incertidumbre** (pocas visitas) en lugar de explorar al azar.

In [ ]:
partidas = 500
turnos = 200
valores_c = [0.5, 1.0, 2.0]

nombres_ucb = [f'UCB (c={c})' for c in valores_c] + ['ε-greedy (ε=0.1)']
recompensas_ucb = np.zeros((len(nombres_ucb), turnos))
optimas_ucb = np.zeros((len(nombres_ucb), turnos))

for partida in range(partidas):
    for i, c in enumerate(valores_c):
        Q = {k: 0 for k in range(tablero.num_casillas)}
        N = {k: 0 for k in range(tablero.num_casillas)}

        for turno in range(turnos):
            mejor_valor = -1e9
            for j in range(tablero.num_casillas):
                if N[j] == 0:
                    valor_ucb = 1e9
                else:
                    valor_ucb = Q[j] + c * math.sqrt(math.log(turno + 1) / N[j])
                if valor_ucb > mejor_valor:
                    mejor_valor = valor_ucb
                    casilla = j

            N[casilla] += 1
            recompensa = tablero.cavar(casilla)
            Q[casilla] += (recompensa - Q[casilla]) / N[casilla]

            recompensas_ucb[i][turno] += recompensa
            optimas_ucb[i][turno] += (1 if casilla == tablero.mejor_casilla else 0)

    # ε-greedy referencia
    Q = {k: 0 for k in range(tablero.num_casillas)}
    N = {k: 0 for k in range(tablero.num_casillas)}
    for turno in range(turnos):
        if np.random.uniform() < 0.1:
            casilla = np.random.randint(tablero.num_casillas)
        else:
            max_q = -1e9
            for j in range(tablero.num_casillas):
                if Q[j] > max_q:
                    max_q = Q[j]
                    casilla = j
        N[casilla] += 1
        r = tablero.cavar(casilla)
        Q[casilla] += (r - Q[casilla]) / N[casilla]
        recompensas_ucb[len(valores_c)][turno] += r
        optimas_ucb[len(valores_c)][turno] += (1 if casilla == tablero.mejor_casilla else 0)

recompensas_ucb /= partidas
optimas_ucb /= partidas

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for i, nombre in enumerate(nombres_ucb):
    plt.plot(recompensas_ucb[i], label=nombre)
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Recompensa promedio')
plt.title('UCB: Recompensa')

plt.subplot(1, 2, 2)
for i, nombre in enumerate(nombres_ucb):
    plt.plot(optimas_ucb[i], label=nombre)
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Proporción óptima')
plt.title('UCB: Acción óptima')
plt.tight_layout()
plt.show()

### Experimento 3: Algoritmo de Gradiente (Softmax)

Asigna **preferencias** $H(a)$ a cada casilla y las convierte en probabilidades con softmax:
$$\pi_t(a) = \frac{e^{H_t(a)}}{\sum_{b} e^{H_t(b)}}$$

Actualiza las preferencias comparando la recompensa con su promedio histórico.

In [ ]:
def softmax(x):
    return np.exp(x) / sum(np.exp(x))

In [ ]:
partidas = 500
turnos = 200
alphas_grad = [0.05, 0.1, 0.3]

nombres_grad = [f'Gradiente (α={a})' for a in alphas_grad] + ['ε-greedy (ε=0.1)']
recompensas_grad = np.zeros((len(nombres_grad), turnos))
optimas_grad = np.zeros((len(nombres_grad), turnos))

for partida in range(partidas):
    for i, alfa in enumerate(alphas_grad):
        H = np.zeros(tablero.num_casillas)
        historial_r = []

        for turno in range(turnos):
            probs = softmax(H)
            casilla = np.random.choice(tablero.num_casillas, p=probs)

            r = tablero.cavar(casilla)
            historial_r.append(r)
            r_prom = np.mean(historial_r)

            for j in range(tablero.num_casillas):
                if j == casilla:
                    H[j] += alfa * (r - r_prom) * (1 - probs[j])
                else:
                    H[j] -= alfa * (r - r_prom) * probs[j]

            recompensas_grad[i][turno] += r
            optimas_grad[i][turno] += (1 if casilla == tablero.mejor_casilla else 0)

    # ε-greedy referencia
    Q = {k: 0 for k in range(tablero.num_casillas)}
    N = {k: 0 for k in range(tablero.num_casillas)}
    for turno in range(turnos):
        if np.random.uniform() < 0.1:
            casilla = np.random.randint(tablero.num_casillas)
        else:
            max_q = -1e9
            for j in range(tablero.num_casillas):
                if Q[j] > max_q:
                    max_q = Q[j]
                    casilla = j
        N[casilla] += 1
        r = tablero.cavar(casilla)
        Q[casilla] += (r - Q[casilla]) / N[casilla]
        recompensas_grad[len(alphas_grad)][turno] += r
        optimas_grad[len(alphas_grad)][turno] += (1 if casilla == tablero.mejor_casilla else 0)

recompensas_grad /= partidas
optimas_grad /= partidas

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for i, nombre in enumerate(nombres_grad):
    plt.plot(recompensas_grad[i], label=nombre)
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Recompensa promedio')
plt.title('Gradiente: Recompensa')

plt.subplot(1, 2, 2)
for i, nombre in enumerate(nombres_grad):
    plt.plot(optimas_grad[i], label=nombre)
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Proporción óptima')
plt.title('Gradiente: Acción óptima')
plt.tight_layout()
plt.show()

### Comparación final

In [ ]:
partidas = 500
turnos = 200
nombres_final = ['ε-greedy (ε=0.1)', 'UCB (c=1.0)', 'Gradiente (α=0.1)']
recompensas_final = np.zeros((3, turnos))
optimas_final = np.zeros((3, turnos))

for partida in range(partidas):
    # ε-greedy
    Q = {k: 0 for k in range(tablero.num_casillas)}
    N = {k: 0 for k in range(tablero.num_casillas)}
    for turno in range(turnos):
        if np.random.uniform() < 0.1:
            c = np.random.randint(tablero.num_casillas)
        else:
            mq = -1e9
            for j in range(tablero.num_casillas):
                if Q[j] > mq:
                    mq = Q[j]
                    c = j
        N[c] += 1
        r = tablero.cavar(c)
        Q[c] += (r - Q[c]) / N[c]
        recompensas_final[0][turno] += r
        optimas_final[0][turno] += (1 if c == tablero.mejor_casilla else 0)

    # UCB
    Q = {k: 0 for k in range(tablero.num_casillas)}
    N = {k: 0 for k in range(tablero.num_casillas)}
    for turno in range(turnos):
        mv = -1e9
        for j in range(tablero.num_casillas):
            if N[j] == 0:
                vu = 1e9
            else:
                vu = Q[j] + 1.0 * math.sqrt(math.log(turno + 1) / N[j])
            if vu > mv:
                mv = vu
                c = j
        N[c] += 1
        r = tablero.cavar(c)
        Q[c] += (r - Q[c]) / N[c]
        recompensas_final[1][turno] += r
        optimas_final[1][turno] += (1 if c == tablero.mejor_casilla else 0)

    # Gradiente
    H = np.zeros(tablero.num_casillas)
    hr = []
    for turno in range(turnos):
        probs = softmax(H)
        c = np.random.choice(tablero.num_casillas, p=probs)
        r = tablero.cavar(c)
        hr.append(r)
        rp = np.mean(hr)
        for j in range(tablero.num_casillas):
            if j == c:
                H[j] += 0.1 * (r - rp) * (1 - probs[j])
            else:
                H[j] -= 0.1 * (r - rp) * probs[j]
        recompensas_final[2][turno] += r
        optimas_final[2][turno] += (1 if c == tablero.mejor_casilla else 0)

recompensas_final /= partidas
optimas_final /= partidas

In [ ]:
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
for i, m in enumerate(nombres_final):
    plt.plot(recompensas_final[i], label=m)
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Recompensa promedio')
plt.title('Recompensa')

plt.subplot(1, 3, 2)
for i, m in enumerate(nombres_final):
    plt.plot(optimas_final[i], label=m)
plt.legend()
plt.grid(True)
plt.xlabel('Turnos')
plt.ylabel('Proporción óptima')
plt.title('Acción óptima')

plt.subplot(1, 3, 3)
x = np.arange(3)
ancho = 0.35
fr = [recompensas_final[i][-1] for i in range(3)]
fo = [optimas_final[i][-1] for i in range(3)]
plt.bar(x - ancho/2, fr, ancho, label='Recompensa final')
plt.bar(x + ancho/2, fo, ancho, label='% Óptima final')
plt.xticks(x, nombres_final, rotation=15)
plt.legend()
plt.title('Comparación final')

plt.tight_layout()
plt.show()

---
## Resumen

### Componentes del problema

| Componente | Clase / Elemento | Descripción |
|---|---|---|
| **Entorno** | `Tablero` | 25 casillas con probabilidades $p_i$ de tener moneda |
| **Estado** | `casilla` | Índice de la casilla (0 a 24) |
| **Acción** | `elegir_casilla()` | El agente decide qué casilla cavar |
| **Política** | ε-greedy / UCB / Gradiente | Estrategia para elegir casillas |
| **Recompensa** | `cavar()` → 1 o 0 | +1 si hay moneda, 0 si no |
| **Valor** | `funcion_de_valor[casilla]` | Estimación de la probabilidad de tener moneda |

### Métodos implementados

1. **ε-greedy**: Clásico, explora al azar con probabilidad ε
2. **UCB**: Explora según incertidumbre $c\sqrt{\ln(t)/N}$
3. **Gradiente (Softmax)**: Preferencias probabilísticas con softmax

Los experimentos usan la estructura de `02_bandits.ipynb` con bucles `for` anidados, y el entrenamiento con clases sigue la lógica de `01_introduccion.ipynb`.